# Lab 5A — Reinforcement Learning in a Nutshell
## From Tables to Neural Networks: MODEL → SAMPLE → SCALE

---

### 🎯 Lab Philosophy

This is the hands-on companion to **Lecture 5 (RL in a Nutshell)**. The lecture had one
map — *replace the model you must **specify** with experience you can **sample**, then
**scale** it with function approximation*. This lab makes that map runnable:

| Act | Lecture deck | What you build here |
|---|---|---|
| **5.1 MODEL** | T1 — RL Framework & DP | a small **economic gridworld**, then the exact optimum by **Value Iteration** (full backup, *needs the model*) |
| **5.2 SAMPLE** | T2 — MC / TD / Q-learning | **Monte Carlo** and **Q-learning** that learn the *same* optimum from experience — *no model* — plus **ε-greedy vs Boltzmann** exploration |
| **5.3 SCALE** | T3 — Actor–Critic & Deep RL | swap the table for a **neural network**: a mini **DQN** (experience replay + target net), then a small **actor–critic** on a consumption–savings problem |

Everything is validated against a **known benchmark** — the Value-Iteration solution
(Parts 2–3.1) and the closed-form log-utility rule (Part 3.2). That validation discipline
*is* the point: an RL result is only credible against something you already trust.

> **The one caveat (T3):** RL scales the *solution*, not the *identification*. Here we can
> check every learner against the exact answer precisely because the model is small and known.

---

### 🌐 Running this notebook (offline-safe)

- Runs **top-to-bottom with no internet**. Only `numpy`, `matplotlib`, and `torch` (CPU is fine).
- If a package is missing, install from the **Tsinghua mirror** in VS Code's terminal:
  ```
  pip install numpy matplotlib torch -i https://pypi.tuna.tsinghua.edu.cn/simple
  ```
- No Colab, no Google Drive, no downloads. Every cell is deterministic (fixed seeds).


In [ ]:
# --- Setup: imports, reproducibility, CPU-friendly threading ---
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# Tiny models on possibly many-core machines: avoid thread oversubscription.
torch.set_num_threads(1)

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cpu")  # everything here is small; CPU is plenty
print("numpy", np.__version__, "| torch", torch.__version__, "| device:", device)

# Part 1 — MODEL: the environment and the DP ground truth

*Act 5.1: when you **have** the model ($P$, $R$), classical DP computes the exact optimum by a
**full-expectation backup**. We build that gold standard first so every later learner has
something to be graded against.*

### The 3×3 economic gridworld (Lec05 T2)

A policymaker navigates **inflation × unemployment**. Each cell is a macro state; the
bottom-right cell $G$ is the target (low inflation, low unemployment). Four actions are
policy interventions (Up = contractionary, Down = expansionary, Left/Right = fiscal /
structural). Policy has its **intended** effect most of the time but **slips** with a small
probability — so this is a genuinely *stochastic* MDP. Reward is $+1$ on reaching $G$, else $0$.


In [ ]:
# --- The gridworld MDP: exposes BOTH the sampled interface (step) and the model (P, R) ---
class GridWorld:
    """A small stochastic gridworld phrased as a macro-policy MDP.

    States: cells of an n x n grid, index s = row*n + col. Goal G = bottom-right (absorbing).
    Actions: 0=Up, 1=Down, 2=Left, 3=Right (walls keep you in place).
    Dynamics: intended move w.p. (1-slip); otherwise a uniformly random action's move.
    Reward: +1 on arriving at the goal, 0 otherwise.
    """
    def __init__(self, n=3, slip=0.1, gamma=0.95):
        self.n, self.slip, self.gamma = n, slip, gamma
        self.nS, self.nA = n * n, 4
        self.start, self.goal = 0, n * n - 1
        self.moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # Up, Down, Left, Right
        self.action_names = ["Up (contractionary)", "Down (expansionary)", "Left", "Right"]
        self._build_model()

    def _rc(self, s):  return divmod(s, self.n)
    def _idx(self, r, c): return r * self.n + c

    def _dest(self, s, a):
        r, c = self._rc(s); dr, dc = self.moves[a]
        nr, nc = r + dr, c + dc
        return self._idx(nr, nc) if (0 <= nr < self.n and 0 <= nc < self.n) else s

    def _build_model(self):
        """Fill P[s,a,s'] and expected reward R[s,a] — the 'model' DP needs."""
        self.P = np.zeros((self.nS, self.nA, self.nS))
        self.R = np.zeros((self.nS, self.nA))
        for s in range(self.nS):
            if s == self.goal:                 # goal is absorbing, zero reward
                self.P[s, :, s] = 1.0
                continue
            for a in range(self.nA):
                self.P[s, a, self._dest(s, a)] += (1 - self.slip)          # intended
                for a2 in range(self.nA):                                   # slip mass
                    self.P[s, a, self._dest(s, a2)] += self.slip / self.nA
                self.R[s, a] = self.P[s, a, self.goal] * 1.0               # E[reward]

    def sample_step(self, s, a, rng):
        """Sampled interface used by the model-FREE learners (Parts 2-3)."""
        sp = int(rng.choice(self.nS, p=self.P[s, a]))
        r = 1.0 if sp == self.goal else 0.0
        return sp, r, (sp == self.goal)

env = GridWorld(n=3, slip=0.1, gamma=0.95)
print(f"states={env.nS}, actions={env.nA}, start={env.start}, goal={env.goal}, gamma={env.gamma}")
assert np.allclose(env.P.sum(2), 1.0), "each P[s,a,.] must be a probability distribution"
print("Model P, R built and validated (rows of P sum to 1).")

### Value Iteration — the full-expectation backup

Because we know $P$ and $R$, we can iterate the Bellman optimality operator to convergence:
$$ (TV)(s) = \max_a \Big\{ R(s,a) + \gamma \sum_{s'} P(s'\mid s,a)\, V(s') \Big\}. $$
By the contraction-mapping theorem (T1), this converges to the unique $V^\*$. This is the
**"needs $P,R$"** band on the map — the answer we will later reach *without* the model.

In [ ]:
# --- Value Iteration: exact optimum via full backups over the known model ---
def value_iteration(env, tol=1e-12, max_iter=10_000):
    V = np.zeros(env.nS)
    for _ in range(max_iter):
        Q = env.R + env.gamma * (env.P @ V)   # (nS,nA); matmul contracts s'
        Vn = Q.max(axis=1); Vn[env.goal] = 0.0
        if np.max(np.abs(Vn - V)) < tol:
            V = Vn; break
        V = Vn
    Q = env.R + env.gamma * (env.P @ V)
    pi = Q.argmax(axis=1)
    return V, Q, pi

def policy_value(env, pi):
    """Exact value of a deterministic policy pi (solve V = R^pi + gamma P^pi V).
    Used only as a *diagnostic* to grade the model-free learners below."""
    idx = np.arange(env.nS)
    Ppi, Rpi = env.P[idx, pi], env.R[idx, pi]
    V = np.linalg.solve(np.eye(env.nS) - env.gamma * Ppi, Rpi)
    V[env.goal] = 0.0
    return V

Vstar, Qstar, pistar = value_iteration(env)
print("V*(s) on the grid (goal = 0):")
print(np.round(Vstar.reshape(env.n, env.n), 3))
code_map = {0: "U", 1: "D", 2: "L", 3: "R"}
grid = [("G" if s == env.goal else code_map[int(pistar[s])]) for s in range(env.nS)]
print("\nOptimal action per state (U/D/L/R, G=goal; several ties exist):")
print(np.array(grid).reshape(env.n, env.n))

In [ ]:
# --- Helper: draw a policy (arrows) over a value heatmap. Reused throughout. ---
def show_policy(env, V, pi, title):
    n = env.n
    fig, ax = plt.subplots(figsize=(4.2, 4.2))
    ax.imshow(V.reshape(n, n), cmap="YlGn")
    arrow = {0: (0, -0.3), 1: (0, 0.3), 2: (-0.3, 0), 3: (0.3, 0)}  # U,D,L,R
    for s in range(env.nS):
        r, c = divmod(s, n)
        if s == env.goal:
            ax.text(c, r, "G", ha="center", va="center", fontsize=16, fontweight="bold")
        else:
            dx, dy = arrow[int(pi[s])]
            ax.arrow(c, r, dx, dy, head_width=0.12, color="navy", length_includes_head=True)
        ax.text(c, r + 0.34, f"{V[s]:.2f}", ha="center", va="center", fontsize=7, color="black")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title)
    plt.tight_layout(); plt.show()

show_policy(env, Vstar, pistar, "Value Iteration: V* and $\\pi^*$ (the benchmark)")

# Part 2 — SAMPLE: learn the same optimum with no model

*Act 5.2: throw the model away. The agent now sees only transitions $(s, a, r, s')$ from
`env.sample_step`. We recover $V^\*$ two ways — **Monte Carlo** (whole episodes) and
**Q-learning** (every step) — the two ways to replace the full backup with a **sample backup**.
Both are **Generalized Policy Iteration**: evaluate, then act greedily, repeat.*

In [ ]:
# --- Shared helpers for the model-free learners ---
def eps_greedy(Q, s, eps, rng):
    if rng.random() < eps:
        return int(rng.integers(Q.shape[1]))
    q = Q[s]
    return int(rng.choice(np.flatnonzero(q == q.max())))   # random tie-break

def random_start(env, rng):
    s = int(rng.integers(env.nS))
    while s == env.goal:
        s = int(rng.integers(env.nS))
    return s                                                 # "exploring starts" (T2)

def greedy_metrics(Q, env, Vstar):
    """Grade a learned Q against the DP benchmark by VALUE, not action identity.
      sup_err : sup-norm gap between the learned greedy value and V* (estimation error).
      val_gap : how much true value you LOSE by following the learned greedy policy
                (max_s |V^pi - V*|). This is the economically meaningful score: the grid
                has several equally-good paths, so which specific action you pick does not
                matter — only the value it delivers does."""
    V = Q.max(axis=1); V[env.goal] = 0.0
    sup_err = float(np.max(np.abs(V - Vstar)))
    val_gap = float(np.max(np.abs(policy_value(env, Q.argmax(axis=1)) - Vstar)))
    return sup_err, val_gap

### Monte Carlo control (first-visit, ε-greedy)

Simulate a full episode, compute the realized return $G_t = \sum_k \gamma^k r_{t+k+1}$, and
move each $Q(s,a)$ toward the returns it actually earned. We use **exploring starts** and a
**GLIE** schedule ($\varepsilon \to 0$) so the greedy policy converges to $\pi^\*$.

> **How we grade (by value, not by action).** Each learner reports two numbers:
> `sup|V-V*|` (how well it estimates the value function) and the **greedy-policy value gap**
> — how much true value you would *lose* by following its policy (≈ 0 means optimal). Because
> several grid paths are equally good, *which* action a learner picks is the wrong yardstick;
> the value it delivers is the right one.

In [ ]:
# --- First-visit Monte Carlo control ---
def mc_control(env, n_episodes=20_000, eps0=0.3, seed=0, max_steps=200,
               Vstar=None, eval_every=200):
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.nS, env.nA)); N = np.zeros((env.nS, env.nA))
    curve = []
    for ep in range(1, n_episodes + 1):
        eps = eps0 * 2000 / (2000 + ep)                      # GLIE decay
        s = random_start(env, rng); traj = []
        for _ in range(max_steps):
            a = eps_greedy(Q, s, eps, rng)
            sp, r, done = env.sample_step(s, a, rng)
            traj.append((s, a, r)); s = sp
            if done: break
        G = 0.0; Gs = [0.0] * len(traj)                      # returns-to-go
        for t in reversed(range(len(traj))):
            G = traj[t][2] + env.gamma * G; Gs[t] = G
        seen = set()
        for t, (s, a, _) in enumerate(traj):                 # first-visit updates
            if (s, a) not in seen:
                seen.add((s, a)); N[s, a] += 1
                Q[s, a] += (Gs[t] - Q[s, a]) / N[s, a]
        if Vstar is not None and ep % eval_every == 0:
            curve.append((ep, greedy_metrics(Q, env, Vstar)[0]))
    return Q, np.array(curve)

Q_mc, curve_mc = mc_control(env, Vstar=Vstar, seed=0)
err_mc, gap_mc = greedy_metrics(Q_mc, env, Vstar)
print(f"Monte Carlo:  sup|V-V*| = {err_mc:.4f}   greedy-policy value gap = {gap_mc:.4f}")

### Temporal-Difference control: Q-learning

Update **every step** toward a *bootstrapped* target $r + \gamma \max_{a'} Q(s',a')$ — sampling
(like MC) **and** bootstrapping (like DP). Off-policy: it learns $Q^\*$ while exploring.

In [ ]:
# --- Q-learning (off-policy TD control) ---
def q_learning(env, n_episodes=20_000, alpha=0.2, eps0=0.3, seed=0, max_steps=200,
               Vstar=None, eval_every=200):
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.nS, env.nA)); curve = []
    for ep in range(1, n_episodes + 1):
        eps = eps0 * 2000 / (2000 + ep)
        s = random_start(env, rng)
        for _ in range(max_steps):
            a = eps_greedy(Q, s, eps, rng)
            sp, r, done = env.sample_step(s, a, rng)
            target = r + (0.0 if done else env.gamma * Q[sp].max())
            Q[s, a] += alpha * (target - Q[s, a])
            s = sp
            if done: break
        if Vstar is not None and ep % eval_every == 0:
            curve.append((ep, greedy_metrics(Q, env, Vstar)[0]))
    return Q, np.array(curve)

Q_ql, curve_ql = q_learning(env, Vstar=Vstar, seed=0)
err_ql, gap_ql = greedy_metrics(Q_ql, env, Vstar)
print(f"Q-learning:   sup|V-V*| = {err_ql:.4f}   greedy-policy value gap = {gap_ql:.4f}")
show_policy(env, Q_ql.max(1), Q_ql.argmax(1), "Q-learning recovers $\\pi^*$ — with no model")

In [ ]:
# --- Both sample-based learners converge to the DP benchmark ---
plt.figure(figsize=(6.4, 3.8))
plt.plot(curve_mc[:, 0], curve_mc[:, 1], label="Monte Carlo (per episode)")
plt.plot(curve_ql[:, 0], curve_ql[:, 1], label="Q-learning / TD (per step)")
plt.axhline(0, color="green", ls="--", lw=1, label="Value Iteration (exact)")
plt.xlabel("episodes"); plt.ylabel("sup$|V_{greedy}-V^*|$")
plt.title("SAMPLE backup converges to the FULL backup")
plt.legend(); plt.tight_layout(); plt.show()

### Exploration: ε-greedy vs Boltzmann (softmax)

To *learn* a good policy you must sometimes act sub-optimally on purpose. Two ways to explore,
reproducing the numbers from the T2 slide with action values $\hat Q = (10, 9, 1)$:
- **ε-greedy** spreads exploration **uniformly** — the near-optimal action and the clearly-bad
  action get the *same* probability.
- **Boltzmann** respects the **value ordering** — it explores the promising action far more than
  the inferior one. (Same softmax/temperature structure as logit discrete choice, and LLM decoding.)

In [ ]:
# --- Reproduce the T2 exploration table: eps-greedy vs Boltzmann on Q = (10, 9, 1) ---
Qhat = np.array([10.0, 9.0, 1.0])

eps = 0.1
p_eps = np.full(3, eps / 3); p_eps[np.argmax(Qhat)] += 1 - eps

tau = 2.0
p_bolt = np.exp(Qhat / tau); p_bolt /= p_bolt.sum()

print(f"{'action':>8}{'Q':>6}{'eps-greedy(e=.1)':>18}{'Boltzmann(tau=2)':>18}")
for i, name in enumerate(["a1", "a2", "a3"]):
    print(f"{name:>8}{Qhat[i]:>6.0f}{p_eps[i]:>18.2f}{p_bolt[i]:>18.2f}")
print("\n-> eps-greedy gives the near-optimal a2 and the bad a3 the SAME 0.03;"
      "\n   Boltzmann sends ~0.37 to a2 and ~0.01 to a3 — exploration that respects value.")

### Recap — MC vs TD vs DP (the T2 comparison)

| Dimension | Monte Carlo | TD(0) / Q-learning | Dynamic Programming |
|---|---|---|---|
| Needs a model? | **No** | **No** | **Yes** ($P, R$) |
| Bootstrapping | No (full return) | Yes (1-step target) | Yes (full backup) |
| Bias / variance | Unbiased / high | Biased / low | Exact (no sampling) |
| Updates | end of episode | every step | full sweep |

All three found the same policy above — **TD sits between MC and DP**: it samples like MC and
bootstraps like DP. Next we remove the last limitation they share: the **table**.

# Part 3 — SCALE: replace the table with a neural network

*Act 5.3: every learner so far stored **one number per state** — a table. Real economic states
(capital, prices, wealth distributions) are continuous and high-dimensional, so the table
explodes (the curse of dimensionality, T1). Replace it with a **function approximator**.*

### 3.1 A mini-DQN on the gridworld — *table → function*

Feed the state as a one-hot vector into a small MLP $Q_\theta(s)\to$ four action-values, and
train it with the **Q-learning target**. Plugging a neural net straight into Q-learning ignites
the **deadly triad** (off-policy + function approximation + bootstrapping), so we add the two
**DQN** patches (T3):
- **experience replay** — a buffer of past transitions, sampled in minibatches (breaks
  correlation, restores approx. i.i.d.);
- a **target network** $\theta^-$ — a lagged copy used in the target, synced every $C$ steps
  (freezes the bootstrap loop).

It should recover the *same* greedy policy as tabular Q-learning / Value Iteration.

In [ ]:
# --- Mini-DQN: neural Q with experience replay + target network ---
class QNet(nn.Module):
    def __init__(self, nS, nA, h=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(nS, h), nn.ReLU(),
                                 nn.Linear(h, h), nn.ReLU(),
                                 nn.Linear(h, nA))
    def forward(self, x): return self.net(x)

def onehot(states, nS):
    X = torch.zeros(len(states), nS)
    X[torch.arange(len(states)), torch.as_tensor(states, dtype=torch.long)] = 1.0
    return X

def train_dqn(env, n_steps=6000, batch=64, lr=1e-3, buffer_cap=4000,
              target_sync=200, eps0=0.3, seed=0):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    q, qt = QNet(env.nS, env.nA), QNet(env.nS, env.nA)
    qt.load_state_dict(q.state_dict())
    opt = torch.optim.Adam(q.parameters(), lr=lr)
    buf = []
    s = random_start(env, rng)
    for step in range(1, n_steps + 1):
        eps = max(0.05, eps0 * 2000 / (2000 + step))
        if rng.random() < eps:
            a = int(rng.integers(env.nA))
        else:
            with torch.no_grad():
                a = int(q(onehot([s], env.nS)).argmax())
        sp, r, done = env.sample_step(s, a, rng)
        buf.append((s, a, r, sp, float(done)))
        if len(buf) > buffer_cap: buf.pop(0)
        s = random_start(env, rng) if done else sp
        if len(buf) >= batch:                                # learn from a minibatch
            idx = rng.integers(0, len(buf), size=batch)
            b = [buf[i] for i in idx]
            S  = onehot([x[0] for x in b], env.nS)
            A  = torch.as_tensor([x[1] for x in b]).long().unsqueeze(1)
            R_ = torch.as_tensor([x[2] for x in b]).float()
            SP = onehot([x[3] for x in b], env.nS)
            D  = torch.as_tensor([x[4] for x in b]).float()
            with torch.no_grad():
                tgt = R_ + (1 - D) * env.gamma * qt(SP).max(1).values
            pred = q(S).gather(1, A).squeeze(1)
            loss = ((pred - tgt) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
        if step % target_sync == 0:
            qt.load_state_dict(q.state_dict())               # sync target net
    with torch.no_grad():
        Qnn = q(onehot(list(range(env.nS)), env.nS)).numpy()
    return Qnn

Q_dqn = train_dqn(env, seed=0)
err_dqn, gap_dqn = greedy_metrics(Q_dqn, env, Vstar)
print(f"mini-DQN:     sup|V-V*| = {err_dqn:.4f}   greedy-policy value gap = {gap_dqn:.4f}")
print("A 64x64 net with ~%d weights reproduced the tabular policy — no table required."
      % sum(p.numel() for p in QNet(env.nS, env.nA).parameters()))

### 3.2 Actor–critic on a consumption–savings problem — *a taste of deep RL for economics*

Now a **continuous** action on the running economic anchor. A household with wealth $w$ chooses
a **consumption rate** $\varphi = c/w \in (0,1)$; savings earn gross return $R = 1/\beta$, so
$w_{t+1} = R\,(1-\varphi)\,w_t$, and the reward is log utility $\log c = \log\varphi + \log w$.

- **Actor** $\pi_\theta(\varphi\mid \log w)$: a small net outputs the mean of a Gaussian over the
  logit of $\varphi$ (continuous action — no $\max$ over actions to enumerate).
- **Critic** $V_\psi(\log w)$: a small net; its TD error is the **advantage** baseline that cuts
  policy-gradient variance (T3).

**Known benchmark (validation):** with log utility, the optimal rule is to consume a *constant*
fraction $\varphi^\* = 1-\beta$ of wealth, independent of $w$. We check the learned rate against it.


In [ ]:
# --- Actor-critic on the log-utility consumption-savings problem (vectorized) ---
def train_actor_critic(beta=0.8, T=50, B=512, updates=700, lr_a=2e-3, lr_v=1e-2,
                       sigma0=0.5, seed=0):
    torch.manual_seed(seed)
    R, gamma = 1.0 / beta, beta
    actor  = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 1))   # -> logit mean
    critic = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 1))   # -> V(log w)
    opt_a = torch.optim.Adam(actor.parameters(),  lr=lr_a)
    opt_v = torch.optim.Adam(critic.parameters(), lr=lr_v)
    hist = []
    for it in range(updates):
        sigma = max(0.05, sigma0 * (1 - it / updates))         # anneal exploration
        x = torch.rand(B, 1) * 2 - 1                            # start log-wealth ~ U(-1,1): "exploring starts"
        logps, rews, vals = [], [], []
        for _ in range(T):
            mu = actor(x)
            dist = torch.distributions.Normal(mu, sigma)
            u = dist.sample()
            logps.append(dist.log_prob(u).sum(1))
            phi = torch.sigmoid(u).clamp(1e-3, 1 - 1e-3).squeeze(1)
            xt = x.squeeze(1)
            rews.append(torch.log(phi) + xt)                    # log c = log phi + log w
            vals.append(critic(x).squeeze(1))
            x = (np.log(R) + torch.log(1 - phi) + xt).unsqueeze(1).detach().clamp(-20, 20)
        # discounted returns-to-go
        G = torch.zeros(B); rets = []
        for t in reversed(range(T)):
            G = rews[t] + gamma * G; rets.insert(0, G.clone())
        rets = torch.stack(rets); vals = torch.stack(vals); logps = torch.stack(logps)
        adv = (rets - vals).detach()
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)
        (-(logps * adv).mean()).backward(); opt_a.step(); opt_a.zero_grad()
        ((rets.detach() - vals) ** 2).mean().backward(); opt_v.step(); opt_v.zero_grad()
        if (it + 1) % 40 == 0:
            with torch.no_grad():
                phi0 = torch.sigmoid(actor(torch.zeros(1, 1))).item()
            hist.append((it + 1, phi0))
    return actor, beta, np.array(hist)

actor, beta, hist = train_actor_critic(seed=0)
phi_star = 1 - beta
with torch.no_grad():
    learned = {w: torch.sigmoid(actor(torch.tensor([[np.log(w)]], dtype=torch.float32))).item()
               for w in [0.5, 1.0, 2.0]}
print(f"Optimal consumption rate  phi* = 1 - beta = {phi_star:.3f}")
for w, phi in learned.items():
    print(f"  learned phi at wealth w={w:>3}:  {phi:.3f}")
print("\nThe actor learned a near-constant consumption rate close to 1 - beta — "
      "the closed-form log-utility rule, discovered from experience.")

In [ ]:
# --- Watch the learned consumption rate approach the closed-form optimum ---
plt.figure(figsize=(6.4, 3.8))
plt.plot(hist[:, 0], hist[:, 1], marker="o", ms=3, label="learned $\\varphi$ (at $w=1$)")
plt.axhline(phi_star, color="green", ls="--", label="optimal $\\varphi^*=1-\\beta$")
plt.xlabel("actor-critic updates"); plt.ylabel("consumption rate $\\varphi$")
plt.title("Actor-critic recovers the log-utility rule")
plt.legend(); plt.tight_layout(); plt.show()

# Wrap-up — the whole lab in one map

You climbed the same three acts as the lecture, and **every rung was checked against a
benchmark you trust**:

| Act | You built | Validated against |
|---|---|---|
| **5.1 MODEL** | gridworld + Value Iteration (full backup) | — (this *is* the benchmark) |
| **5.2 SAMPLE** | Monte Carlo, Q-learning, ε-greedy vs Boltzmann | matched $V^\*,\pi^\*$ with **no model** |
| **5.3 SCALE** | mini-DQN (replay + target net); actor–critic | DQN matched $\pi^\*$; actor rate → $1-\beta$ |

**One backbone underneath all of it:** Generalized Policy Iteration — evaluate, act greedily,
repeat. Only *how we filled the expectation* changed: full backup → sample backup → sampled +
function-approximated.

### 🧪 Your turn (exercises)
1. **GLIE matters.** Fix $\varepsilon$ (remove the decay) in `q_learning`. Does `policy match` still
   reach 100%? Relate to the T2 convergence condition.
2. **SARSA vs Q-learning.** Implement on-policy SARSA (target $r+\gamma Q(s',a')$ with $a'$ the
   *actually taken* next action) and compare. Which is more conservative near the walls? (T2: safety
   vs optimality.)
3. **Feel the curse.** Rebuild `env = GridWorld(n=8)`. How many table entries now? This is why Part 3
   replaces the table with a net.
4. **Comparative statics.** In `train_actor_critic`, set `beta=0.9`. Does the learned rate track the
   new $1-\beta=0.1$? You just did RL *comparative statics*.

### ➡️ Where this goes next (Lec06)
The actor–critic you just wrote is the seed of the **Session-4 deep-HA lab**: *Krusell–Smith (1998)
via Deep Equilibrium Nets in PyTorch*, where the state includes the **wealth distribution** and the
same value/policy-network idea scales to a heterogeneous-agent economy. **The toolkit doesn't change
— the state space does.**
